## Intro — Araucanía Case Study

This notebook prepares `locations_custom_from_metadata.yaml` for input to the Calliope model.
It uses cluster metadata and reduced links from `003_clustering_Araucania.ipynb`.

Demand: hourly H2 demand timeseries per cluster (from `cluster_h2_demand_profiles.csv`).
Ammonia and electricity supply/conversion techs are enabled at all nodes.
Areas in km², power in MW, costs in k$.

In [1]:
import pandas as pd
import yaml
from tqdm import tqdm

# ── Settings ──────────────────────────────────────────────────────────────────
IDENTIFIER        = "CHI_ARN"
n_clusters_target = '15'   # must match the clustering run

path = (f'../Cluster_Output_Araucania/Agglomerative/'
        f'Agglomerative_k{n_clusters_target}/')

metadata_path     = path + "cluster_metadata.csv"
connections_path  = path + "reduced_links.csv"

cluster_df         = pd.read_csv(metadata_path)
try:
    interconnections_df = pd.read_csv(connections_path)
except pd.errors.EmptyDataError:
    interconnections_df = pd.DataFrame(columns=[
        'cluster_a', 'cluster_b', 'rep_cell_a', 'rep_cell_b', 'n_cross_edges', 'distance_km'
    ])

# Location string used as the node key throughout
cluster_df['location'] = (cluster_df['lat'].round(2).astype(str) + '_' +
                           cluster_df['lon'].round(2).astype(str))

print(f"Loaded {len(cluster_df)} clusters")
print(f"Loaded {len(interconnections_df)} interconnections")

# ── Build demand dictionary ────────────────────────────────────────────────────
# Only H2 demand — timeseries from cluster_h2_demand_profiles.csv
demand_locations_dict = {}
for idx, row in cluster_df.iterrows():
    cluster_id     = int(row['cluster_id'])
    cluster_column = f'cluster_{cluster_id}'
    location_str   = row['location']

    demand_locations_dict[location_str] = {
        'demand_hydrogen': f'file=cluster_h2_demand_profiles.csv:{cluster_column}'
    }

print(f"\nDemand dictionary built for {len(demand_locations_dict)} locations")
print(f"  - H2 demand: time-varying from cluster_h2_demand_profiles.csv")

Loaded 15 clusters
Loaded 25 interconnections

Demand dictionary built for 15 locations
  - H2 demand: time-varying from cluster_h2_demand_profiles.csv


In [2]:
class NoAliasDumper(yaml.SafeDumper):
    def ignore_aliases(self, data):
        return True

def format_region_name(lat_lon):
    return f"region_{lat_lon}".replace('.', '_')

def generate_locations_from_metadata(metadata_df, interconnections_df, demand_locations_dict):
    locations        = {}
    links            = {}
    group_constraints = {}

    wind_profile_file  = 'cluster_wind_profiles_weighted.csv'
    solar_profile_file = 'cluster_solar_profiles_weighted.csv'

    print("Creating locations from metadata...")

    for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Creating Locations"):
        location_str   = row['location']
        cluster_id     = int(row['cluster_id'])
        cluster_column = f'cluster_{cluster_id}'
        lat            = float(row['lat'])
        lon            = float(row['lon'])
        land_area      = float(row['available_land_area_km2'])
        cluster_name   = format_region_name(location_str)

        # Supply and conversion techs — enabled at all nodes
        techs = {
            'onshore_wind':                  {'constraints': {'resource': f'file={wind_profile_file}:{cluster_column}'}},
            'solar_single_axis':             {'constraints': {'resource': f'file={solar_profile_file}:{cluster_column}'}},
            'battery':                       None,
            'electrolyser':                  None,
            'compressed_hydrogen_storage':   None,
            'fuel_cell':                     None,
            'haber_and_air_separation':      None,
            'ammonia_storage':               None,
            'ammonia_ccgt':                  None,
            'hydrogen_ccgt':                 None,
        }

        # Add H2 demand timeseries for this cluster
        if location_str in demand_locations_dict:
            for demand_tech, demand_profile in demand_locations_dict[location_str].items():
                techs[demand_tech] = {'constraints': {'resource': demand_profile}}

        locations[cluster_name] = {
            'coordinates': {'lat': lat, 'lon': lon},
            'techs': techs
        }

        # Land area group constraint (wind + solar share available area)
        group_constraints[f"combined_wind_solar_area_limit_{cluster_name}"] = {
            'techs':              ['onshore_wind', 'solar_single_axis'],
            'locs':               [cluster_name],
            'resource_area_max':  land_area
        }

    print("\nCreating links from interconnections...")

    for idx, row in tqdm(interconnections_df.iterrows(), total=len(interconnections_df), desc="Creating Links"):
        cluster_from = row['cluster_a']
        cluster_to   = row['cluster_b']

        loc_from = metadata_df[metadata_df['cluster_id'] == cluster_from]['location'].values[0]
        loc_to   = metadata_df[metadata_df['cluster_id'] == cluster_to  ]['location'].values[0]

        cluster_from_name = format_region_name(loc_from)
        cluster_to_name   = format_region_name(loc_to)

        link_key     = f"{cluster_from_name},{cluster_to_name}"
        distance_km  = float(row['distance_km'])   # Araucanía column name

        links[link_key] = {
            'techs': {
                'dc_transmission':  {'distance': distance_km},
                'hydrogen_pipeline':{'distance': distance_km},
                'ammonia_pipeline': {'distance': distance_km},
            }
        }

    print(f"\nCreated {len(locations)} locations")
    print(f"Created {len(links)} links")
    print(f"Created {len(group_constraints)} group constraints")

    return locations, links, group_constraints

In [3]:
locations, links, group_constraints = generate_locations_from_metadata(
    metadata_df=cluster_df,
    interconnections_df=interconnections_df,
    demand_locations_dict=demand_locations_dict
)

yaml_string = """
##
# LOCATIONS
##
"""
yaml_string += yaml.dump({'locations': locations}, Dumper=NoAliasDumper, default_flow_style=False)

yaml_string += """
##
# TRANSMISSION CAPACITIES
##
"""
yaml_string += yaml.dump({'links': links}, Dumper=NoAliasDumper, default_flow_style=False)

yaml_string += """
##
# GROUP CONSTRAINTS
##
"""
yaml_string += yaml.dump({'group_constraints': group_constraints}, Dumper=NoAliasDumper, default_flow_style=False)

output_filename = path + "locations_custom_from_metadata.yaml"

print(f"Writing YAML to {output_filename}...")
with open(output_filename, 'w') as f:
    for line in tqdm(yaml_string.splitlines(), desc="Writing YAML"):
        f.write(line + '\n')

print(f"\n✓ {output_filename} generated successfully!")
print(f"  - {len(locations)} locations")
print(f"  - {len(links)} interconnections")
print(f"  - {len(group_constraints)} land area constraints")

Creating locations from metadata...


Creating Locations: 100%|██████████| 15/15 [00:00<?, ?it/s]



Creating links from interconnections...


Creating Links: 100%|██████████| 25/25 [00:00<00:00, 748.65it/s]


Created 15 locations
Created 25 links
Created 15 group constraints


Writing YAML to ../Cluster_Output_Araucania/Agglomerative/Agglomerative_k15/locations_custom_from_metadata.yaml...


Writing YAML: 100%|██████████| 650/650 [00:00<?, ?it/s]


✓ ../Cluster_Output_Araucania/Agglomerative/Agglomerative_k15/locations_custom_from_metadata.yaml generated successfully!
  - 15 locations
  - 25 interconnections
  - 15 land area constraints
